# Spotify Music Dashboard (2023)

Interactive dashboard of the most streamed Spotify songs in 2023. Use the **Released year** dropdown to explore top songs, release timing, playlist reach, and audio features from 2011–2023. Switch between **Light** and **Dark** themes with the theme dropdown.


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import interact

df = pd.read_csv('spotify-2023.csv', encoding='cp1252')
df = df.drop(columns=['in_shazam_charts', 'key'])
df = df[df['streams'] != 'BPM110KeyAModeMajorDanceability53Valence75Energy69Acousticness7Instrumentalness0Liveness17Speechiness3']
df['streams'] = df['streams'].astype('int')

THEME_TEMPLATES = {
    'Light': 'plotly_white',
    'Dark': 'plotly_dark',
}


In [ ]:
def top_stream_counts(df, year, theme='Light'):
    this_year_df = df[df['released_year']==year][['track_name', 'artist(s)_name', 'streams']]
    this_year_sorted = this_year_df.sort_values(by='streams', ascending=False)
    top10 = this_year_sorted.head(10)
    top10_sorted = top10.sort_values(by='streams', ascending=True)

    fig = px.bar(
        top10_sorted,
        y = 'track_name',
        x = 'streams',
        hover_data='artist(s)_name',
        title = 'Top 10 most streamed songs released in ' + str(year),
        text_auto = True,
        orientation = 'h'
    )
    fig.update_layout(
        height=700, # give each bar enough thickness
        bargap=0.2, 
        yaxis_title="",
        xaxis_title="Stream Count",
        template=THEME_TEMPLATES[theme],
        margin=dict(l=10, r=20, t=50, b=40)
    )
    #fig.show()
    return fig

In [ ]:
def stream_vs_playlist(df, year, theme='Light'):
    this_year_df = df[df['released_year']==year][['track_name', 'artist(s)_name', 'streams', 'in_spotify_charts', 'in_spotify_playlists']]
    
    fig = px.scatter(
            this_year_df, 
            x='in_spotify_playlists', 
            y='streams',
            opacity=0.7,                 
            size='in_spotify_charts',                       
            hover_name='track_name',          
            hover_data=['artist(s)_name', 'in_spotify_charts'],
            title='Streams vs. Number of Spotify playlists of top songs released in ' + str(year)
        )
    
    fig.update_layout(
        template=THEME_TEMPLATES[theme],
        xaxis_title='Number of Spotify playlists',
        yaxis_title='Stream Count',
    )
    #fig.show()
    return fig

In [ ]:
def released_month_hist(df, year, theme='Light'):
    this_year_df = df[df['released_year']==year]['released_month']

    fig = px.histogram(
        this_year_df,
        x = 'released_month',
        nbins = 12,
        title = 'Number of most streamed songs released each month in ' + str(year),
        text_auto = True
    )
    fig.update_xaxes(
        dtick=1,                            
        tickvals=list(range(1, 13)),       
        ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']        
    )                           
    fig.update_layout(
        xaxis_title='Released month in ' + str(year),
        yaxis_title='Number of most streamed songs in 2023',
        bargap=0.2,
        template=THEME_TEMPLATES[theme]
    )
    
    #fig.show()
    return fig

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def dve_vplot(df, year, theme='Light'):
    cols = ['danceability_%', 'valence_%', 'energy_%']
    this_year_df = df[df['released_year']==year][cols]

    fig = make_subplots(
        rows=1, 
        cols=3, 
        subplot_titles=['Danceability', 'Valence', 'Energy'],
        shared_yaxes=True
    )
    for i, col in enumerate(cols, start=1):
        fig.add_trace(
            go.Violin(
                y=this_year_df[col],
                name=col.replace('_%', ' percentage').capitalize(),
                box_visible=True,            
                spanmode='hard', # to prevent plots go beyond 100%                   
                showlegend=False
            ),
            row=1, col=i
        )
    fig.update_yaxes(range=[0, 105], title_text="Percentage (%)", row=1, col=1)
    fig.update_layout(
        title='Audio feature distributions of top songs released in ' + str(year),
        template=THEME_TEMPLATES[theme],
        height=500,
        margin=dict(t=80, b=40, l=60, r=40)
    )
    
    #fig.show()
    return fig

In [ ]:
def create_dashboard(df, year, theme='Light'):
    ax1 = top_stream_counts(df, year, theme)
    ax2 = released_month_hist(df, year, theme)
    ax3 = stream_vs_playlist(df, year, theme)
    ax4 = dve_vplot(df, year, theme)

    titles = [
        ax1.layout.title.text if ax1.layout.title.text else "Top Stream Counts",
        ax2.layout.title.text if ax2.layout.title.text else "Streams vs Playlist",
        ax3.layout.title.text if ax3.layout.title.text else "Release Month Distribution",
        ax4.layout.title.text if ax4.layout.title.text else "Audio Features"
    ]

    dashboard = make_subplots(
        rows=2, 
        cols=2,
        subplot_titles=titles,
        horizontal_spacing=0.12,
        vertical_spacing=0.15
    )

    fig_map = [
        (ax1, 1, 1),
        (ax2, 1, 2),
        (ax3, 2, 1),
        (ax4, 2, 2)
    ]

    for fig, r, c in fig_map:
        for trace in fig.data:
            dashboard.add_trace(trace, row=r, col=c)

    # this is to fix x-label on histogram
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
                   
    dashboard.update_xaxes(title_text="Stream Count", row=1, col=1)
    dashboard.update_yaxes(title_text="", row=1, col=1)

    dashboard.update_xaxes(
        dtick=1,
        tickvals=list(range(1, 13)),
        ticktext=month_names,
        range=[0.5, 12.5],
        title_text=f'Released month in {year}',
        row=1, col=2
    )
    dashboard.update_yaxes(title_text='Number of most streamed songs in 2023', row=1, col=2)

    dashboard.update_xaxes(title_text='Number of Spotify playlists', row=2, col=1)
    dashboard.update_yaxes(title_text='Stream Count', row=2, col=1)

    dashboard.update_yaxes(title_text='Percentage (%)', row=2, col=2)

    dashboard.update_layout(
        title_text=f"Spotify Music Dashboard ({year})",
        title_font_size=20,
        height=900,
        width=1700,
        template=THEME_TEMPLATES[theme],
        showlegend=False,
        bargap=0.15,  # <--- Restores spacing between bars globally across subplots
        margin=dict(t=100, b=50, l=50, r=50)
    )

    dashboard.show()            

In [ ]:
year_options = list(range(2011, 2024))

@interact(
    year=widgets.Dropdown(
        options=year_options,
        value=2023,
        description='Released year: ',
        style={'description_width': 'initial'}
    ),
    theme=widgets.Dropdown(
        options=list(THEME_TEMPLATES.keys()),
        value='Light',
        description='Theme: ',
        style={'description_width': 'initial'}
    )
)
def interactive_dashboard(year, theme):
    create_dashboard(df, year, theme)